In [ ]:
REPO_URL = "https://github.com/KietAnhCS/BTS3D-GS.git"
REPO_DIR = "/content/BTS3D-GS"
REPO_SUBDIR = None

import os, sys

if os.path.isdir("pipeline"):
    TARGET_DIR = os.getcwd()
else:
    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    TARGET_DIR = os.path.join(REPO_DIR, REPO_SUBDIR) if REPO_SUBDIR else REPO_DIR

%cd {TARGET_DIR}
sys.path.insert(0, TARGET_DIR)

In [ ]:
from pipeline.env import check_gpu, install_dependecies
from pipeline.data import mount drive 

install_dependecies()
mount_drive()
check_gpu(require=True)

In [ ]:
from pipeline import Config, run, report 

cfg = Config(
    drive_mount=True,
    scene_root="/content/drive/MyDrive/VAI_NVS_DATA_ROUND2",
    drive_subdir="HCM0539",        # chỉ dùng scene này; None = lấy hết
    dataset_url=None,

    data_root="/content/data",
    scenes=(),                     # () = mọi scene tìm thấy dưới scene_root
    resolution=2,                  # ảnh train giảm 2x cho vừa T4
    iterations=30000,
    score_every=1000,              # tần suất chấm điểm khi train
    eval_views=None,               # chấm trên toàn bộ ảnh hold-out, không lấy mẫu con
    psnr_max=30.0,                 # PSNR_max của ban tổ chức
    submission_resolution=1,       # render đúng kích thước ảnh gốc

    # --- chống mất model khi phiên Colab bị ngắt ---------------------
    autosave_to_drive=True,        # chép .ply sang Drive NGAY sau mỗi scene
    drive_run_dir="sadgs_runs",    # -> /content/drive/MyDrive/sadgs_runs/<scene>/
    save_to_drive=True,            # submission.zip + models.zip cũng lên Drive
    run_smoke=True,                # False = Run All bỏ qua bước chạy thử

    # --- submission ---------------------------------------------------
    submission_order="csv", 
)

# --- Cách 2: tải bằng gdown (thay 4 dòng đầu của Config ở trên) -------
#     drive_mount=False,
#     scene_root=None,
#     drive_folder_url="https://drive.google.com/drive/folders/1QqEkZevaEaB7NYZwfE--KlDnn21B3qFz",
#     drive_subdir=None,           # link trên đã trỏ thẳng HCM0539 rồi
#
# Link thư mục cha (mọi scene) — dùng kèm drive_subdir="HCM0539":
#   https://drive.google.com/drive/folders/1zeov1zNpcmgTit9IuJnH38OEAFt79BXI

cfg.show()

In [ ]:
scenes, profile = run.load_data(cfg)
display(profile)

In [ ]:
run.smoke_test(cfg, scenes[0])

In [ ]:
results, submission = run.run_all(cfg, scenes)

In [ ]:
history, board = run.analytics(cfg, results, submissions)
display(board)

In [ ]:
report.show_samples(cfg, scene[0], n=3)

In [ ]:
check, problems = run.finish(cfg, scenes)
display(check)